# Embeddings
After tokenizing, we are left with arbitrary integers that do not have any true mathematical meeting. The solution is to map each token id to a vector of floats called embedding. These embeddings are learned: they start random and are updating throughout training.

As an overly simple example, we can imagine that each vector has two dimensions: [x,y]. We can imagine. that after training, tokens with similar meanings end up in similar spaces. So "duck" and "goose" would exist closer than "soap" and "pillar". Furthermore, we can also infer syntactic meanings based on the 'distances' between words: for instance, we might imagine that in embeddings space, "king" - "queen" ~ "man" - "woman".

In reality, we use many more dimensions than two but we cannot visualize more than three.

**(Credit for code, structure, explanations, and examples: [raiyanyahya](https://github.com/raiyanyahya/how-to-train-your-gpt/blob/master/chapters/03_embeddings.md)!)**

### Additional Context
1. Initialize embedding table randomly
2. Training: compute loss and update according to backprop
* using example from rayanyahya: during training, model sees 'the cat sat on the mat'. But when it predicts something else instead of 'mat', the loss is high. So backprop sends a signal: embedding for 'cat' should be updated so it is more predictive of 'mat', and embedding for 'mat' shold be updated so it is closer to things that follow 'the'
3. Emergent structure: after training on billions of tokens, we see a meaningful structure emerge.

In [1]:
import torch
import torch.nn as nn
import math

# lookup
class Embedding(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    # call parent class
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embedding_dim) #nn.Embedding is an optimized lookup table; tensor of token_ids --> corr_rows.
    self.embedding_dim = embedding_dim

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    embeddings = self.embed(x)
    #scale_factor = math.sqrt(self.embedding_dim)
    scale_factor = 1.0
    embeddings_scaled = embeddings * scale_factor
    return embeddings_scaled

# Elaboration on this code
### Pseudocode
```
EMBEDDING:
create embedding table of shape (vocab_size, embedding_dim)

FORWARD:
input: input tokens of shape (batch_size, sequence_length)
for each token_id:
  look up its row in the embedding table (nn.Embedding does this simultaneously in one operation)
return matrix of shape (batch_size, sequence_length, embedding_dim)

...
nn.Embedding:
initialize self.weight with a random matrix of size (vocab_size, embeddings_dim)
forward: return self.weight[x] where

weight: given token_ids, return weight[token] for token in token_ids (simultaneously)
```

### Why Scale Embeddings?

Some Transformer implementations multiply token embeddings by `sqrt(d_model)` before adding positional encodings:

$$
x = \sqrt{d_{\text{model}}} \cdot E[\text{token}] + PE[\text{position}]
$$

The reason is that token embeddings and positional encodings are added together, so their **relative scale** matters.

If the embedding vectors are initialized with small per-coordinate variance, roughly:

$$
\operatorname{Var}(E_i) \approx \frac{1}{d_{\text{model}}}
$$

then each coordinate has standard deviation:

$$
\operatorname{Std}(E_i) \approx \frac{1}{\sqrt{d_{\text{model}}}}
$$

That means the raw embedding values are very small. Multiplying by `sqrt(d_model)` brings them back to order-1 scale:

$$
\operatorname{Std}\left(\sqrt{d_{\text{model}}} \cdot E_i\right) \approx 1
$$

Now the token embedding signal is on a similar scale to sinusoidal positional encodings, whose values are usually between `-1` and `1`.

So the scaling helps prevent one signal from completely dominating the other when we compute:

$$
\text{input} = \text{token embedding} + \text{positional encoding}
$$

However, this explanation only applies if the embedding table was initialized with small variance, such as standard deviation:

$$
\frac{1}{\sqrt{d_{\text{model}}}}
$$

If the embedding table is initialized as:

$$
N(0, 1)
$$

as PyTorch `nn.Embedding` often is by default, then multiplying by `sqrt(d_model)` makes the token embeddings much larger than the positional encodings.

For example, with:

$$
d_{\text{model}} = 768
$$

we have:

$$
\sqrt{768} \approx 27.7
$$

So embeddings with standard deviation `1` become embeddings with standard deviation about `27.7`, while positional encodings are still only between `-1` and `1`.

In that case, scaling by `sqrt(d_model)` is not needed unless the embedding weights are reinitialized to have smaller variance.

In [3]:
embed = Embedding(50267, 768)
example = torch.tensor([[[42, 6429, 284, 651, 1365, 379, 3047, 616, 4981], [47991, 250, 166, 113, 255, 168, 244, 112, 3823]]])
print(f"Input shape (batch_size, sequence_length): {example.shape}")
fwd = embed.forward(example)
print(f"Output shape (batch_size, sequence_length, embedding_dim): {fwd.shape}")

Input shape (batch_size, sequence_length): torch.Size([1, 2, 9])
Output shape (batch_size, sequence_length, embedding_dim): torch.Size([1, 2, 9, 768])
